# NB-07 — End-to-End Inference Pipeline

**Goal:** Use `MultimodalQAPipeline` as a single entry point for real multimodal QA.

Supports: file paths, URLs, base64, PIL images, multi-turn history, text-only fallback, and optional streaming.

---

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import time
import base64
from io import BytesIO

import torch
from dotenv import load_dotenv
from PIL import Image

load_dotenv(Path("..") / ".env")

from src.pipeline.multimodal_qa import ImageLoadError, MultimodalQAPipeline

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
CONFIG = Path("..") / "config" / "model_config.yaml"
print(f"Device: {DEVICE}")

## 1. Load the full pipeline

First run downloads CLIP + Qwen2-VL-2B (~4GB). On Apple Silicon, `encoder_on_cpu=True` saves MPS memory for the LLM.

In [ ]:
print("Loading MultimodalQAPipeline...")
pipeline = MultimodalQAPipeline(
    CONFIG,
    device=DEVICE,
    encoder_on_cpu=(DEVICE == "mps"),
)
print(f"Max new tokens: {pipeline.max_new_tokens}")

## 2. Demo — single image QA (URL input)

In [ ]:
IMG_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "4/43/Cute_dog.jpg/320px-Cute_dog.jpg"
)
question = "What animal is in this image? Answer in one short sentence."

t0 = time.perf_counter()
answer = pipeline.answer(question, image=IMG_URL)
latency = time.perf_counter() - t0

print(f"Latency: {latency:.2f}s")
print(f"Answer: {answer}")

## 3. Demo — multi-turn conversation (same image)

History is a list of `{"role": "user"|"assistant", "content": str}` turns.

In [ ]:
history: list[dict[str, str]] = []
rounds = [
    "What is the main subject?",
    "What colors do you notice?",
    "Would this be a good pet photo? Why?",
]

for i, q in enumerate(rounds, 1):
    reply = pipeline.answer(q, image=IMG_URL, history=history, max_new_tokens=128)
    print(f"--- Turn {i} ---")
    print(f"User: {q}")
    print(f"Assistant: {reply}\n")
    history.append({"role": "user", "content": q})
    history.append({"role": "assistant", "content": reply})

## 4. Demo — text-only (no image)

The custom path still runs through the language model with text embeddings only.

In [ ]:
text_only = pipeline.answer(
    "Explain contrastive learning in one paragraph.",
    image=None,
    max_new_tokens=200,
)
print(text_only)

## 5. Input types — path, PIL, base64

In [ ]:
pil_img = pipeline._load_image(IMG_URL)
cache_path = Path("..") / "data" / "nb07_sample.jpg"
cache_path.parent.mkdir(parents=True, exist_ok=True)
pil_img.save(cache_path)

buf = BytesIO()
pil_img.save(buf, format="JPEG")
b64 = base64.b64encode(buf.getvalue()).decode("ascii")

for label, src in [("path", str(cache_path)), ("PIL", pil_img), ("base64", b64)]:
    loaded = pipeline._load_image(src)
    print(f"{label}: {loaded.size} {loaded.mode}")

## 6. Edge cases and error handling

In [ ]:
def try_answer(label: str, **kwargs) -> None:
    try:
        out = pipeline.answer(**kwargs, max_new_tokens=64)
        print(f"[{label}] OK — {str(out)[:120]}...")
    except (ImageLoadError, ValueError) as exc:
        print(f"[{label}] Expected error: {exc}")

try_answer("bad path", question="What?", image="/tmp/not_a_real_image.jpg")
try_answer("empty question", question="   ", image=None)

long_q = "What do you see? " + "Please be detailed. " * 80
try_answer("very long question", question=long_q, image=IMG_URL)

## 7. Native baseline vs custom pipeline

Compare Qwen2-VL's built-in vision tower (`use_native=True`) with our CLIP + MLP projector path.

In [ ]:
q = "Describe the scene briefly."

custom = pipeline.answer(q, image=IMG_URL, use_native=False, max_new_tokens=128)
native = pipeline.answer(q, image=IMG_URL, use_native=True, max_new_tokens=128)

print("Custom (CLIP+MLP):", custom)
print("Native (Qwen2-VL):", native)

## 8. Projector comparison (Linear vs MLP vs Q-Former)

Each variant reloads the model — run only when you have time/GPU memory. Q-Former uses fewer visual tokens (32 queries).

In [ ]:
from omegaconf import OmegaConf
from src.llm.backbone import MultimodalLLM

COMPARE = False  # set True to run side-by-side projector comparison

if COMPARE:
    base_cfg = OmegaConf.load(CONFIG)
    pil = pipeline._load_image(IMG_URL)
    test_q = "What is in this image?"

    for ptype in ("linear", "mlp", "qformer"):
        cfg = base_cfg.copy()
        cfg.projector.type = ptype
        tmp = Path("..") / "config" / f"_nb07_{ptype}.yaml"
        OmegaConf.save(cfg, tmp)
        print(f"\n=== Projector: {ptype} ===")
        m = MultimodalLLM.from_config(tmp, device=DEVICE, encoder_on_cpu=(DEVICE == "mps"))
        ans = m.generate(pil, test_q, max_new_tokens=96)
        print(ans)
else:
    print("Set COMPARE=True to reload models for linear / mlp / qformer comparison.")

## 9. Streaming (optional)

Custom path currently yields the full answer in one chunk; native streaming can be added in Phase 7 API work.

In [ ]:
chunks = list(pipeline.answer("Name one object.", image=IMG_URL, stream=True, max_new_tokens=64))
print("Streamed chunks:", chunks)

## 10. Optional — Gradio UI

Uncomment and run if `gradio` is installed (`pip install gradio`).

In [ ]:
RUN_GRADIO = False

if RUN_GRADIO:
    import gradio as gr

    def gradio_answer(image, question, history_json):
        hist = history_json or []
        return pipeline.answer(question, image=image, history=hist, max_new_tokens=256)

    demo = gr.Interface(
        fn=gradio_answer,
        inputs=["image", "text", gr.State([])],
        outputs="text",
        title="VisionMind QA",
    )
    demo.launch()
else:
    print("Set RUN_GRADIO=True to launch a local demo.")

---

**Phase 4 complete.** Next: Phase 5 — LoRA fine-tuning (`src/llm/lora_finetune.py`, `NB-08-lora-finetuning.ipynb`).